# Model Server trên Colab (BGE-M3 encode + reranker) qua ngrok

Máy local (GPU 4GB) không load nổi 2 model. Notebook này chạy 2 model nặng trên Colab
GPU T4, expose FastAPI qua ngrok. Máy local gọi `/encode` + `/rerank`, giữ Qdrant +
generation ở máy.

**Trước khi chạy:** Runtime > Change runtime type > **GPU (T4)**.

**Cần:** ngrok authtoken (https://dashboard.ngrok.com/get-started/your-authtoken).

## 1. Cài + cấu hình (bỏ HF_TOKEN lỗi để không treo tải model)

In [ ]:
import os
os.environ.pop('HF_TOKEN', None)                 # tránh token lỗi làm treo tải model
os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'    # tải model nhanh/ổn định hơn

!pip -q install fastapi uvicorn pyngrok hf_transfer FlagEmbedding sentence-transformers

import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2. Load 2 model (1 lần, trên GPU)

In [ ]:
from FlagEmbedding import BGEM3FlagModel
from sentence_transformers import CrossEncoder   # reranker qua CrossEncoder (FlagReranker lỗi transformers 5.x)
print('Load BGE-M3...'); ENCODER = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, devices='cuda')
print('Load reranker...'); RERANKER = CrossEncoder('BAAI/bge-reranker-v2-m3', device='cuda')
print('Sẵn sàng.')

## 3. FastAPI: /encode, /rerank, /health

Bảo vệ bằng header `X-Token` (đặt SECRET dưới, dán vào .env máy local — RAG_REMOTE_TOKEN).

In [ ]:
SECRET = 'doi-token-nay-cho-rieng-ban'   # <-- ĐỔI token này, dán cùng giá trị vào .env máy local

from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel

app = FastAPI()

class EncodeReq(BaseModel):
    query: str
    max_length: int = 1024

class RerankReq(BaseModel):
    query: str
    texts: list[str]

def _check(tok):
    if tok != SECRET:
        raise HTTPException(status_code=401, detail='bad token')

@app.get('/health')
def health():
    return {'ok': True}

@app.post('/encode')
def encode(req: EncodeReq, x_token: str = Header(None)):
    _check(x_token)
    out = ENCODER.encode([req.query], max_length=req.max_length,
                         return_dense=True, return_sparse=True, return_colbert_vecs=False)
    sw = out['lexical_weights'][0]
    return {'dense': [float(x) for x in out['dense_vecs'][0]],
            'sparse': {'indices': [int(k) for k in sw.keys()],
                       'values': [float(v) for v in sw.values()]}}

@app.post('/rerank')
def rerank(req: RerankReq, x_token: str = Header(None)):
    _check(x_token)
    if not req.texts:
        return {'scores': []}
    scores = RERANKER.predict([[req.query, t] for t in req.texts])   # CrossEncoder API
    return {'scores': [float(s) for s in scores]}

## 4. Mở ngrok + chạy server (giữ cell này CHẠY suốt phiên)

In [ ]:
from pyngrok import ngrok
import uvicorn, threading, time

ngrok.set_auth_token('DAN_NGROK_AUTHTOKEN_VAO_DAY')   # <-- authtoken ngrok của bạn

# chạy server 1 LẦN duy nhất (dùng biến toàn cục đánh dấu) -> chạy lại cell không lỗi port
if globals().get('_SERVER_STARTED'):
    print('[server] đã chạy rồi. URL:', globals().get('_PUBLIC_URL'))
else:
    ngrok.kill()                         # giết agent ngrok cũ (tránh ERR_NGROK_334 "already online")
    try:
        for t in ngrok.get_tunnels():
            ngrok.disconnect(t.public_url)
    except Exception:
        pass
    _PUBLIC_URL = ngrok.connect(8000).public_url

    def _run():
        uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
    threading.Thread(target=_run, daemon=True).start()
    _SERVER_STARTED = True
    time.sleep(3)

print('=== NGROK URL (dán vào .env máy local: RAG_REMOTE_URL) ===')
print(_PUBLIC_URL)
print('=== RAG_REMOTE_TOKEN =', SECRET, '===')
print('\n[server] GIỮ notebook mở. Test:  !curl -s', _PUBLIC_URL + '/health')